# Dataset Code Smells
Lê automaticamente todos os arquivos `.json` da pasta `/content/smells` e classifica os code smells por classe em formato binário.

In [4]:
import json
import pandas as pd
import os
import glob

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [5]:
SMELLS_FOLDER = "/content/smells"   # pasta com os arquivos JSON
OUTPUT_CSV    = "dataset_code_smells.csv"  # arquivo de saída

In [6]:
# Busca todos os arquivos .json na pasta
json_files = sorted(glob.glob(os.path.join(SMELLS_FOLDER, "*.json")))

if not json_files:
    raise FileNotFoundError(f"Nenhum arquivo .json encontrado em: {SMELLS_FOLDER}")

print(f"{len(json_files)} arquivo(s) encontrado(s) em '{SMELLS_FOLDER}':\n")
for f in json_files:
    print(f"   → {os.path.basename(f)}")

# Carrega cada JSON e usa o nome do arquivo (sem .json) como nome do projeto
all_data = {}
for filepath in json_files:
    project_name = os.path.splitext(os.path.basename(filepath))[0]
    with open(filepath, "r", encoding="utf-8") as f:
        all_data[project_name] = json.load(f)

print(f"\n{len(all_data)} projeto(s) carregado(s)")

11 arquivo(s) encontrado(s) em '/content/smells':

   → BGPP-develop.json
   → Rivfader-master.json
   → Sugarscape-master.json
   → XMPP-Client-master.json
   → cmu-commons-master-indri.json
   → cmu-commons-master-mtj.json
   → interviews-master.json
   → java-design-patterns-master.json
   → mall-master.json
   → rgb-master.json
   → smells_sugarscape-master.json

11 projeto(s) carregado(s)


In [7]:
def collect_all_smells(data):
    """Varre um JSON e retorna o conjunto de smells únicos."""
    smells = set()
    for cls in data:
        for smell in cls.get("smells", []):
            smells.add(smell["name"])
        for method in cls.get("methods", []):
            for smell in method.get("smells", []):
                smells.add(smell["name"])
    return smells


# União de todos os smells de todos os projetos
all_smells_global = set()
for nome, d in all_data.items():
    found = collect_all_smells(d)
    all_smells_global.update(found)
    print(f"   {nome:<45} smells: {sorted(found) if found else '(nenhum)'}")

all_smells_global = sorted(all_smells_global)

print(f"\nTotal de smells únicos em todos os projetos: {len(all_smells_global)}")
for s in all_smells_global:
    print(f"   → {s}")

   BGPP-develop                                  smells: ['ClassDataShouldBePrivate', 'DataClass', 'FeatureEnvy', 'IntensiveCoupling', 'LazyClass', 'LongMethod', 'LongParameterList', 'MessageChain']
   Rivfader-master                               smells: ['ComplexClass', 'FeatureEnvy', 'IntensiveCoupling', 'LazyClass', 'LongMethod', 'MessageChain', 'RefusedBequest', 'SpaghettiCode']
   Sugarscape-master                             smells: ['ClassDataShouldBePrivate', 'ComplexClass', 'DataClass', 'FeatureEnvy', 'IntensiveCoupling', 'LazyClass', 'LongMethod', 'LongParameterList', 'SpaghettiCode']
   XMPP-Client-master                            smells: ['ClassDataShouldBePrivate', 'ComplexClass', 'DataClass', 'DispersedCoupling', 'FeatureEnvy', 'GodClass', 'IntensiveCoupling', 'LazyClass', 'LongMethod', 'LongParameterList', 'MessageChain', 'SpaghettiCode', 'SpeculativeGenerality']
   cmu-commons-master-indri                      smells: ['BrainMethod', 'ComplexClass', 'DataClass', 'Feat

In [8]:
def extract_path_info(path):
    """Extrai package_name e class_name do caminho do arquivo."""
    if not path:
        return "", ""
    path = path.replace("\\", "/")
    filename = os.path.basename(path)
    class_name = filename.replace(".java", "")
    parts = path.split("/")
    package_name = parts[-2] if len(parts) >= 2 else ""
    return package_name, class_name


def get_smells_set(entry):
    """Coleta todos os smells de uma classe (nível classe + nível método)."""
    smells = set()
    for smell in entry.get("smells", []):
        smells.add(smell["name"])
    for method in entry.get("methods", []):
        for smell in method.get("smells", []):
            smells.add(smell["name"])
    return smells


rows = []

for project_name, data in all_data.items():
    for cls in data:
        path = cls.get("sourceFile", {}).get("file", {}).get("path", "")
        package_name, class_name = extract_path_info(path)

        fqn = cls.get("fullyQualifiedName")
        if fqn:
            class_name = fqn

        smells_found = get_smells_set(cls)

        row = {
            "project_name": project_name,
            "package_name": package_name,
            "class_name"  : class_name,
        }

        # Uma coluna por smell — 1 se presente, 0 se não
        for smell in all_smells_global:
            row[smell] = 1 if smell in smells_found else 0

        rows.append(row)


df = pd.DataFrame(rows)
print(f"DataFrame criado: {df.shape[0]} linhas × {df.shape[1]} colunas")
df.head(10)

DataFrame criado: 5201 linhas × 19 colunas


,project_name,package_name,class_name,BrainMethod,ClassDataShouldBePrivate,ComplexClass,DataClass,DispersedCoupling,FeatureEnvy,GodClass,IntensiveCoupling,LazyClass,LongMethod,LongParameterList,MessageChain,RefusedBequest,ShotgunSurgery,SpaghettiCode,SpeculativeGenerality
0,BGPP-develop,Controllers,Controllers.AddEditReservationController,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0
1,BGPP-develop,Controllers,Controllers.AddEditReservationController.Cance...,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
2,BGPP-develop,Controllers,Controllers.AddEditReservationController.Delet...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,BGPP-develop,Controllers,Controllers.AddEditReservationController.SaveL...,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0
4,BGPP-develop,Controllers,Controllers.CustomerOverviewController,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,BGPP-develop,Controllers,Controllers.CustomerOverviewController.SearchL...,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
6,BGPP-develop,Controllers,Controllers.CustomerOverviewController.TableCl...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,BGPP-develop,Controllers,Controllers.EditCustomerController,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
8,BGPP-develop,Controllers,Controllers.EditCustomerController.SaveListener,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9,BGPP-develop,Controllers,Controllers.EditCustomerController.CancelListener,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0


In [9]:
print("Total de classes com cada smell (todos os projetos):\n")
for smell in all_smells_global:
    total = df[smell].sum()
    pct   = total / len(df) * 100
    print(f"   {smell:<35} {total:>4} classes  ({pct:.1f}%)")

print(f"\n   {'Total de classes analisadas':<35} {len(df):>4}")

Total de classes com cada smell (todos os projetos):

   BrainMethod                            3 classes  (0.1%)
   ClassDataShouldBePrivate              73 classes  (1.4%)
   ComplexClass                          58 classes  (1.1%)
   DataClass                             99 classes  (1.9%)
   DispersedCoupling                      4 classes  (0.1%)
   FeatureEnvy                          171 classes  (3.3%)
   GodClass                              36 classes  (0.7%)
   IntensiveCoupling                     25 classes  (0.5%)
   LazyClass                            845 classes  (16.2%)
   LongMethod                            99 classes  (1.9%)
   LongParameterList                    348 classes  (6.7%)
   MessageChain                         314 classes  (6.0%)
   RefusedBequest                        58 classes  (1.1%)
   ShotgunSurgery                         3 classes  (0.1%)
   SpaghettiCode                         14 classes  (0.3%)
   SpeculativeGenerality                209 c

In [10]:
resumo = df.groupby("project_name")[all_smells_global].sum()
resumo["total_classes"] = df.groupby("project_name")["class_name"].count()
resumo["classes_com_smell"] = df.groupby("project_name")[all_smells_global].apply(lambda x: (x.sum(axis=1) > 0).sum())
print("Smells por projeto:")
resumo

Smells por projeto:


,BrainMethod,ClassDataShouldBePrivate,ComplexClass,DataClass,DispersedCoupling,FeatureEnvy,GodClass,IntensiveCoupling,LazyClass,LongMethod,LongParameterList,MessageChain,RefusedBequest,ShotgunSurgery,SpaghettiCode,SpeculativeGenerality,total_classes,classes_com_smell
project_name,,,,,,,,,,,,,,,,,,
BGPP-develop,0,1,0,2,0,10,0,2,8,4,4,4,0,0,0,0,38,28
Rivfader-master,0,0,1,0,0,25,0,1,32,10,0,10,9,0,1,0,131,73
Sugarscape-master,0,7,1,1,0,2,0,1,3,2,1,0,0,0,1,0,12,9
XMPP-Client-master,0,3,4,3,1,10,1,4,19,22,18,31,0,0,3,1,81,62
cmu-commons-master-indri,1,0,1,1,0,1,0,0,3,1,2,1,1,0,0,3,63,12
cmu-commons-master-mtj,0,0,1,0,0,0,0,2,4,1,0,0,1,0,0,4,19,10
interviews-master,1,6,11,0,0,0,0,0,50,1,25,0,0,0,0,0,286,88
java-design-patterns-master,1,44,14,12,0,82,0,1,534,26,90,228,46,1,2,125,3752,1109
mall-master,0,0,5,78,3,20,32,6,176,11,198,24,1,0,2,76,752,494


In [11]:
mask = df[all_smells_global].sum(axis=1) > 0
df_com_smells = df[mask]

print(f"Classes com pelo menos 1 smell: {len(df_com_smells)} de {len(df)}")
df_com_smells

Classes com pelo menos 1 smell: 1940 de 5201


,project_name,package_name,class_name,BrainMethod,ClassDataShouldBePrivate,ComplexClass,DataClass,DispersedCoupling,FeatureEnvy,GodClass,IntensiveCoupling,LazyClass,LongMethod,LongParameterList,MessageChain,RefusedBequest,ShotgunSurgery,SpaghettiCode,SpeculativeGenerality
0,BGPP-develop,Controllers,Controllers.AddEditReservationController,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0
1,BGPP-develop,Controllers,Controllers.AddEditReservationController.Cance...,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
3,BGPP-develop,Controllers,Controllers.AddEditReservationController.SaveL...,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0
5,BGPP-develop,Controllers,Controllers.CustomerOverviewController.SearchL...,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
7,BGPP-develop,Controllers,Controllers.EditCustomerController,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5194,smells_sugarscape-master,scr,scr.SCChartPopulation,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5195,smells_sugarscape-master,scr,scr.SCChartPopulationAge,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5196,smells_sugarscape-master,scr,scr.SCChartSugar,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0
5197,smells_sugarscape-master,scr,scr.SCChartWealth,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [12]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"Arquivo salvo: {OUTPUT_CSV}")
print(f"   Projetos : {list(all_data.keys())}")
print(f"   Linhas   : {len(df)}")
print(f"   Colunas  : {list(df.columns)}")

Arquivo salvo: dataset_code_smells.csv
   Projetos : ['BGPP-develop', 'Rivfader-master', 'Sugarscape-master', 'XMPP-Client-master', 'cmu-commons-master-indri', 'cmu-commons-master-mtj', 'interviews-master', 'java-design-patterns-master', 'mall-master', 'rgb-master', 'smells_sugarscape-master']
   Linhas   : 5201
   Colunas  : ['project_name', 'package_name', 'class_name', 'BrainMethod', 'ClassDataShouldBePrivate', 'ComplexClass', 'DataClass', 'DispersedCoupling', 'FeatureEnvy', 'GodClass', 'IntensiveCoupling', 'LazyClass', 'LongMethod', 'LongParameterList', 'MessageChain', 'RefusedBequest', 'ShotgunSurgery', 'SpaghettiCode', 'SpeculativeGenerality']
